# Summary of evaluation of CPM hourly emulators

In [ ]:
%reload_ext autoreload

%autoreload 2

%reload_ext dotenv
%dotenv

In [ ]:
from mlde_analysis.furflex_default_params import *

In [ ]:
import dask
import dask.array
from dask.distributed import Client
import functools
import IPython
import math
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import string
import xarray as xr

from mlde_analysis import plot_map
from mlde_analysis.display import pretty_table, VAR_RANGES
from mlde_analysis.distribution import mean_bias, std_bias, stat_bias, plot_freq_density, plot_distribution_figure, compute_metrics, DIST_THRESHOLDS, plot_freq_density_figure
from mlde_analysis.wet_dry import threshold_exceeded_prop_stats, THRESHOLDS
from mlde_utils import cp_model_rotated_pole
from mlde_analysis.psd import plot_psd, pysteps_rapsd

In [ ]:
client = Client()
client

In [ ]:
matplotlib.rcParams['figure.dpi'] = 100

In [ ]:
IPython.display.Markdown(desc)

In [ ]:
%%time
%reload_ext mlde_analysis.furflex_magics
EVAL_DS, MODELS, TARGET_DAS, PRED_DAS, VAR_DAS, MODELLABEL2SPEC, EVAL_STATS = %load_eval_data
display(EVAL_DS[target_sim_key])
# EVAL_STATS[target_sim_key]["pred"]["pr"]

## Overall annual distribution

* Frequency Density Histogram of rainfall intensities
* Maps of Mean bias ($\frac{\mu_{sample}-\mu_{CPM}}{\mu_{CPM}}$) over all samples, time and ensemble members
* Std Dev Bias $\frac{\sigma_{sample}}{\sigma_{CPM}}$ over all samples, time and ensemble members

Table of:

* RMS biases
* J-S Distances
* proportion of density over thresholds

In [ ]:
%%time
for var in eval_vars:
    IPython.display.display_markdown(f"### {var}", raw=True)
    
    pred_stats = EVAL_STATS["CPM"]["pred"][var]
    target_stats = EVAL_STATS[target_sim_key]["sim"][var]
    
    metrics_ds = compute_metrics(pred_stats, target_stats, thresholds=DIST_THRESHOLDS[var]).compute()

    pretty_table(metrics_ds, round=4)
    
    normalize=(var == "pr")
    
    def bias(pred_stat, target_stat, normalize):
        raw_bias = pred_stat - target_stat
        if normalize:
            return (
                (100 * raw_bias / target_stats["mean"])
                .rename("Relative bias [%]")
                .assign_attrs({"long_name": "Bias", "units": "%"})
            )
        else:
            return raw_bias.rename(f"Bias [{target_da.attrs['units']}]").assign_attrs(
                {"long_name": "Bias", "units": target_da.attrs["units"]}
            )    
    
    mean_biases = bias(pred_stats["mean"], target_stats["mean"], normalize=normalize).mean(dim="sample_id")
    std_biases = bias(pred_stats["std"], target_stats["std"], normalize=normalize).mean(dim="sample_id")
    q999_biases = bias(pred_stats["q999"], target_stats["q999"], normalize=normalize).mean(dim="sample_id")

    bias_kwargs = {"style": f"{var}Bias"}
    for fd_kwargs in [{"yscale": "log", "target_label": target_sim_key}]:
        fig = plt.figure(layout="constrained", figsize=(5.5, 5))
        
        axd = plot_distribution_figure(
            fig,
            pred_stats, 
            target_stats,
            {"meanb": mean_biases, "stdb": std_biases, "q999b": q999_biases},
            MODELLABEL2SPEC,
            hrange=VAR_RANGES[var],
            fd_kwargs=fd_kwargs,
            bias_kwargs=bias_kwargs
        )
        if var == "relhum150cm":
            axd["Density"].axvline(x=100, color='k', linestyle='--', linewidth=1)
        
        plt.show()

## Threshold exceedence

In [ ]:
%%time
for var, thresholds in THRESHOLDS.items():
    if var in eval_vars:

        target_da = TARGET_DAS[var]
        
        threshold_exceeded_stats = { threshold: threshold_exceeded_prop_stats(VAR_DAS[var][f"pred_{var}"], target_da, threshold) for threshold in thresholds }

        dfs = [
            threshold_exceeded_stats[threshold].cf.mean(["X", "Y"]).to_dataframe().style.set_table_attributes("style='display:inline'").set_caption(f"Threshold: {threshold}{target_da.attrs['units']}").format(precision=1).to_html() 
            for threshold in thresholds
        ]

        IPython.display.display_html(functools.reduce(lambda v, e: v+e, dfs), raw=True)

In [ ]:
client.close()